# 10 · Phiên nghe và đặc trưng last.fm (ticket #55, #56)

Biến 253 triệu lượt nghe last.fm (Music4All-Onion, chưa có khái niệm phiên) thành bảng phiên giống `sessions.parquet` của MPD, để dùng lại `consecutive_pairs`, `smoothness` và điểm mix học (#23) trên bộ thứ hai. Cách cắt ghi ở `src/data/lastfm.py` và `docs/data-contract.md` (mục Bộ phụ last.fm).

**Input** (Add Input): Kaggle Dataset `mixnext-onion` (output notebook 09).

**Output** (`/kaggle/working/out/lastfm/`):
- `sessions.parquet`: `session_id`, `user_id`, `split` (theo user), `pos`, `track_id`
- `lastfm_log.json`: phễu từng bước, độ nhạy theo khoảng nghỉ 15/30/60 phút, thống kê từng split
- `tracks.parquet` (#56): `track_id`, `bpm`, `key`, `mode`, `camelot`, `energy` (thay thế), `key_strength`, `n_occurrences`
- `tracks_log.json` (#56): tự kiểm tra tông so với Essentia, tỉ lệ nghi lỗi nhịp, số bài trưởng/thứ
- `fig_lastfm_features.png` (#56): phân bố BPM, Camelot, energy

Cả hai ticket nằm chung một notebook vì Dataset tạo từ output notebook chỉ nhận version mới từ chính notebook đó.

Chỉ cần CPU, bật **Internet** để clone repo. Nếu thiếu RAM: đặt `SAMPLE_USERS` < 1 (tỉ lệ ghi vào log).

In [ ]:
# Bootstrap: lấy code từ GitHub (#55, #56 đã merge vào main).
BRANCH = "main"
REPO = "/tmp/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
import sys; sys.path.insert(0, REPO)

In [ ]:
from pathlib import Path
ONION = sorted(Path("/kaggle/input").rglob("listening_events.parquet"))[0].parent
OUT = Path("/kaggle/working/out/lastfm")
SAMPLE_USERS = 1.0   # giảm (ví dụ 0.5) nếu hết RAM
print("input:", ONION, sorted(p.name for p in ONION.iterdir()))

In [ ]:
import json, time
from src.data.lastfm import main

t0 = time.time()
log = main(["--onion", str(ONION), "--out", str(OUT), "--sample-users", str(SAMPLE_USERS)])
print(f"{time.time() - t0:.0f} giây")

In [ ]:
# Phễu, độ nhạy theo khoảng nghỉ, các split
import pandas as pd
print(pd.DataFrame(log["funnel"]).to_string(index=False))
print()
print(pd.DataFrame(log["sensitivity"]).T.rename_axis("gap (phút)").to_string())
print()
print(pd.DataFrame(log["splits"]).T.to_string())
sessions = pd.read_parquet(OUT / "sessions.parquet")
lengths = sessions.groupby("session_id").size()
print(f"\nđộ dài phiên: trung vị {lengths.median():.0f}, p90 {lengths.quantile(0.9):.0f}, tối đa {lengths.max()}")

In [ ]:
# #56: đặc trưng cho bộ last.fm (BPM, tông từ HPCP 36 bin gốc La, energy = hạng phần trăm loudness)
from src.features.onion import main as features_main

t0 = time.time()
stats = features_main(["--onion", str(ONION), "--sessions", str(OUT / "sessions.parquet"), "--out", str(OUT)])
print(f"{time.time() - t0:.0f} giây")

In [ ]:
# Phân bố BPM, Camelot, energy (hình cho báo cáo)
import matplotlib.pyplot as plt

tracks = pd.read_parquet(OUT / "tracks.parquet")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(tracks["bpm"], bins=60, color="#4C72B0")
axes[0].set(title="BPM (rhythm.bpm)", xlabel="BPM", ylabel="số bài")
order = [f"{n}{l}" for l in "AB" for n in range(1, 13)]
counts = tracks["camelot"].value_counts().reindex(order, fill_value=0)
axes[1].bar(counts.index, counts.values, color=["#DD8452" if c.endswith("A") else "#55A868" for c in counts.index])
axes[1].set(title="Tông ước lượng (Camelot; cam = thứ, xanh = trưởng)", ylabel="số bài")
axes[1].tick_params(axis="x", rotation=90)
axes[2].hist(tracks["energy"], bins=40, color="#8172B3")
axes[2].set(title="energy thay thế (hạng phần trăm loudness)", xlabel="energy", ylabel="số bài")
fig.tight_layout()
fig.savefig(OUT / "fig_lastfm_features.png", dpi=150)
for name, label in [("key_strength_vs_essentia", "đang dùng (gộp 3s−1..3s+1, gốc La)"),
                    ("check_offset_grouping", "đối chứng: gộp lệch ⅓ cung"),
                    ("check_essentia_like_36", "đối chứng: kiểu Essentia 36 bin")]:
    print(f"{label:38s} Pearson {stats[name]['pearson']:.3f} | lệch trung vị {stats[name]['median_abs_diff']:.3f}")
phase = stats["bin_phase"]
print(f"gốc bin: tâm nốt lệch {phase['center_offset_bins']:+.2f} bin so với bin 3s → "
      f"{'giữ cách gộp có tâm' if phase['centered_grouping_ok'] else 'XEM LẠI gốc bin'} (ngưỡng ±0,5 bin)")
octave = stats["octave_suspect"]
print(f"nghi lỗi nhịp ×2/½: {octave['suspect']:,} bài = {octave['rate_of_second_peak']:.1%} số bài có đỉnh thứ hai, "
      f"{octave['rate_of_all']:.1%} mọi bài")
print("trưởng / thứ / phẳng:", stats["modes"])

## Đăng output

1. **Save Version → Save & Run All (Commit)**.
2. Tab Output → **New Dataset** tên **`mixnext-lastfm`**, **Visibility: Private**, License CC BY 4.0 (dữ liệu gốc Music4All-Onion). Dataset đã có (từ #55): trên trang Dataset chọn **New Version** từ output version mới của notebook này.

## Kết quả #55 (Kaggle CPU, 28/11/2026, toàn bộ user, `SAMPLE_USERS` = 1)

| bước | lượt | phiên | bài | user |
|---|---:|---:|---:|---:|
| đầu vào | 252.984.396 | | 56.512 | 119.140 |
| bỏ timestamp hỏng (trước 2002) | 252.984.395 | | 56.512 | 119.140 |
| cắt theo khoảng nghỉ 30 phút | 252.984.395 | 80.178.838 | 56.512 | 119.140 |
| bỏ bài không có đặc trưng | 252.860.043 | 80.162.092 | 56.463 | 119.140 |
| gộp lặp liên tiếp | 234.774.072 | 80.162.092 | 56.463 | 119.140 |
| giữ phiên ≥ 5 bài | **119.447.177** | **13.292.849** | 56.432 | 105.212 |

| khoảng nghỉ | phiên | cặp liền nhau |
|---:|---:|---:|
| 15 phút | 9.889.636 | 68.330.510 |
| **30 phút** | **13.292.849** | **106.154.328** |
| 60 phút | 14.602.312 | 133.485.717 |

| split | user | phiên | lượt |
|---|---:|---:|---:|
| train | 84.170 | 10.589.365 | 94.953.715 |
| val | 10.521 | 1.358.159 | 12.183.129 |
| test | 10.521 | 1.345.325 | 12.310.333 |

Độ dài phiên: trung vị 7, p90 14, tối đa **30.830**.

- Chỉ 1 lượt có timestamp hỏng (1970). Essentia phủ gần hết: bỏ bài không có đặc trưng chỉ mất 124 nghìn lượt (0,05%).
- Gộp lặp bỏ 18 triệu lượt (7%): người nghe last.fm hay phát lại một bài nhiều lần liền.
- Bước giữ phiên ≥ 5 bài bỏ nhiều nhất (80 triệu phiên → 13,3 triệu): phần lớn phiên rất ngắn. Còn 106 triệu cặp liền nhau để học điểm mix, lớn hơn nhiều so với MPD.
- Số cặp tăng mạnh theo khoảng nghỉ (68 → 106 → 133 triệu cho 15 → 30 → 60 phút): ngưỡng 30 phút là một lựa chọn, báo cáo nên nêu.
- Chia theo user đúng 80/10/10 về số user.
- **Lưu ý cho #57:** có phiên dài bất thường (tối đa 30.830 bài, nghe liên tục nhiều ngày không nghỉ quá 30 phút), có thể là phần mềm tự scrobble hoặc radio bật suốt. Khi bootstrap theo phiên, những phiên này nặng ngang hàng nghìn phiên thường; #57 nên cân nhắc giới hạn độ dài phiên hoặc báo cả kết quả khi bỏ chúng.

## Kết quả #56 (Kaggle CPU, 28/11/2026, 45 giây cho phần đặc trưng)

Phần cắt phiên chạy lại ra đúng các số của #55 ở trên (tất định).

| | giá trị |
|---|---:|
| số bài trong `tracks.parquet` | 56.432 |
| trưởng / thứ / phẳng | 31.549 / 24.883 / 0 |
| tâm nốt lệch so với bin 3s | **+0,20 bin** (ngưỡng ±0,5 → giữ cách gộp có tâm) |
| tỉ lệ bài theo pha 0 / 1 / 2 | 61,2% / 35,3% / 3,6% |
| nghi lỗi nhịp ×2 hoặc ½ | 24 bài (0,05% số bài có đỉnh thứ hai; 46.940 bài có đỉnh thứ hai) |

| `key_strength` so với `tonal.key_krumhansl.strength` | Pearson | lệch trung vị |
|---|---:|---:|
| **đang dùng** (gộp 3s−1..3s+1, gốc La) | **0,705** | 0,083 |
| đối chứng: gộp lệch ⅓ cung (3s..3s+2) | 0,645 | 0,076 |
| đối chứng: kiểu Essentia 36 bin, không cộng hòa âm | 0,754 | 0,049 |

- Gốc bin (bin 0 = La) lấy từ mã nguồn Essentia; trên 56.432 bài, phép kiểm tra có ngưỡng xác nhận: năng lượng nốt dồn ở bin 3s (61%) và 3s+1 (35%), gần như không ở 3s+2 (3,6%), tâm trung bình +0,20 bin (tức khoảng +0,07 cung, nhạc hơi cao hơn tần số lên dây ước tính).
- Trên đủ dữ liệu, cách gộp đang dùng tương quan với Essentia cao hơn cách gộp lệch (0,705 so với 0,645), ngược với mẫu 20 bài thử (0,29 so với 0,55): mẫu thử quá nhỏ. Kiểu Essentia 36 bin cao nhất (0,754) vì gần phương pháp của Essentia nhất; phần còn lại do Essentia cộng thêm hòa âm. Tương quan 0,7 chưa phải bằng hệt: tông tự ước lượng vẫn là một giới hạn cần nêu ở #57 và báo cáo.
- Nghi lỗi nhịp gấp đôi/một nửa rất hiếm (24 bài), nên BPM của Essentia dùng thẳng được cho luật BPM (luật đã tính cả ×2/½).
- Hình phân bố: `fig_lastfm_features.png` trong Dataset `mixnext-lastfm`.